In [1]:
import json, re, random
import pandas as pd
from sklearn.model_selection import train_test_split

with open("../data/raw/cuad-main/data/CUADv1.json") as f:
    data = json.load(f)

rows = []
for contract in data["data"]:
    title = contract["title"]
    para = contract["paragraphs"][0]
    for qa in para["qas"]:
        m = re.search(r'related to "([^"]+)"', qa["question"])
        label = m.group(1) if m else qa["question"]
        for ans in qa["answers"]:                 # loop over ALL answers
            text = re.sub(r"\s+", " ", ans["text"]).strip()
            if text:
                rows.append({"contract": title, "clause_text": text, "label": label})

df = pd.DataFrame(rows)
print(len(df), "rows,", df.label.nunique(), "labels")

13823 rows, 41 labels


In [2]:
freq = df["label"].value_counts()

def pick_rarest(labels):
    # if a clause has several valid labels, keep the rarest one
    # so small classes aren't starved further
    return min(labels, key=lambda l: (freq[l], l))

grouped = (df.groupby(["contract", "clause_text"])["label"]
             .agg(lambda s: sorted(set(s)))
             .reset_index())
grouped["all_labels"] = grouped["label"].apply(lambda ls: ";".join(ls))
grouped["label"] = grouped["label"].apply(pick_rarest)

df = grouped[["contract", "clause_text", "label", "all_labels"]]
print(len(df), "rows after dedup,", (df.all_labels.str.contains(";")).sum(), "still multi-label")

12391 rows after dedup, 1210 still multi-label


In [3]:
random.seed(42)
other_rows = []

for contract in data["data"]:
    title = contract["title"]
    para = contract["paragraphs"][0]
    context = para["context"]
    answer_spans = [(a["answer_start"], a["answer_start"] + len(a["text"]))
                     for qa in para["qas"] for a in qa["answers"]]

    candidates = []
    for m in re.finditer(r"\S(?:.*?\S)?(?=\n\s*\n|\Z)", context, flags=re.S):
        start, end = m.start(), m.end()
        text = re.sub(r"\s+", " ", m.group()).strip()
        if not 15 <= len(text.split()) <= 200:
            continue
        if any(start < b and a < end for a, b in answer_spans):
            continue
        candidates.append(text)

    for text in random.sample(candidates, min(3, len(candidates))):
        other_rows.append({"contract": title, "clause_text": text,
                            "label": "Other", "all_labels": "Other"})

df = pd.concat([df, pd.DataFrame(other_rows)], ignore_index=True)
print(len(df), "rows,", df.label.nunique(), "labels")

13718 rows, 42 labels


In [4]:
contracts = df["contract"].unique()
train_c, temp_c = train_test_split(contracts, test_size=0.2, random_state=42)
val_c, test_c = train_test_split(temp_c, test_size=0.5, random_state=42)

train_df = df[df.contract.isin(train_c)]
val_df   = df[df.contract.isin(val_c)]
test_df  = df[df.contract.isin(test_c)]

train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

# one weight per label: rarer labels get a bigger weight
counts = train_df["label"].value_counts()
weights = (len(train_df) / (train_df["label"].nunique() * counts)).to_dict()
with open("../data/processed/class_weights.json", "w") as f:
    json.dump(weights, f, indent=2)

print(len(train_df), len(val_df), len(test_df))
print("smallest class:", counts.idxmin(), counts.min())
print("largest class:", counts.idxmax(), counts.max())

10953 1253 1512
smallest class: Unlimited/All-You-Can-Eat-License 23
largest class: Parties 2045
